In [ ]:
!pip install -q kaggle
!kaggle datasets download -d zalando-research/fashionmnist
!unzip fashionmnist.zip -d fashionmnist/
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import os
import kagglehub
from matplotlib import pyplot as plt

Dataset URL: https://www.kaggle.com/datasets/zalando-research/fashionmnist
License(s): other
fashionmnist.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  fashionmnist.zip
replace fashionmnist/fashion-mnist_test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
train_df = pd.read_csv('fashionmnist/fashion-mnist_train.csv')
test_df = pd.read_csv('fashionmnist/fashion-mnist_test.csv')

In [ ]:
def image_from_row(row, df=train_df):
    return 2*df.iloc[row, 1:].values.reshape(28,28,1)/255 -1

def images_from_df(indices, df=train_df):
    return 2*df.iloc[indices, 1:].values.reshape(len(indices), 28,28,1)/255 -1

In [ ]:
images = images_from_df([0,1,2,3,4,5,6,7,8,9], train_df)
for image_index in range(10):
    plt.figure()
    plt.imshow(images[image_index], cmap="gray")
    plt.show()

In [ ]:
from tensorflow.keras import models, layers
critico = models.Sequential()
critico.add(layers.Conv2D(32, (3, 3), activation='leaky_relu', input_shape=(28, 28, 1)))
critico.add(layers.Conv2D(64, (3, 3), activation='leaky_relu'))
critico.add(layers.MaxPooling2D((2, 2)))
critico.add(layers.Conv2D(128, (3, 3), activation='leaky_relu'))
critico.add(layers.MaxPooling2D((2, 2)))
critico.add(layers.Flatten())
critico.add(layers.Dense(256, activation='leaky_relu'))
critico.add(layers.Dense(1))
critico.summary()

In [ ]:
gerador = models.Sequential()
gerador.add(layers.Conv2DTranspose(64, (5, 5), activation='leaky_relu', padding="same", input_shape=(4, 4, 8)))
gerador.add(layers.BatchNormalization())
gerador.add(layers.Conv2DTranspose(64, (5, 5), activation='leaky_relu', padding="same"))
gerador.add(layers.BatchNormalization())
gerador.add(layers.Conv2DTranspose(32, (5, 5), activation='leaky_relu', padding="same", strides=3))
gerador.add(layers.BatchNormalization())
gerador.add(layers.Conv2DTranspose(16, (6, 6), activation='leaky_relu', strides=2))
gerador.add(layers.BatchNormalization())
gerador.add(layers.Conv2D(1, (3, 3), activation='tanh', padding="same"))
gerador.summary()

In [ ]:
random_vector = tf.random.normal((1,4,4,8))
print(tf.reduce_max(gerador(random_vector)))
plt.imshow(tf.squeeze(gerador(random_vector)[0]), cmap="gray")

In [ ]:
import math
from tqdm.notebook import tqdm
from keras.losses import BinaryCrossentropy
from keras.optimizers import Adam
os.makedirs('./checkpoints', exist_ok=True)


batch_size = 64
epochs = 50
lambda_gp = 10


optimizer_gerador = Adam(learning_rate=0.0002, beta_1=0.5)
optimizer_critico = Adam(learning_rate=0.0001, beta_1=0.5)


ckpt = tf.train.Checkpoint(
    gerador=gerador,
    critico=critico,
    optimizer_gerador=optimizer_gerador,
    optimizer_critico=optimizer_critico,
    epoch=tf.Variable(0),
    batch=tf.Variable(0)
)

manager = tf.train.CheckpointManager(
    ckpt,
    "./checkpoints",
    max_to_keep=5
)
ckpt.restore(manager.latest_checkpoint)

if manager.latest_checkpoint:
    print("Checkpoint carregado!")
    print("Época:", int(ckpt.epoch))
    print("Batch:", int(ckpt.batch))
else:
    print("Treinamento do zero.")

loss_history = []
grads_history = {}
for epoch in tqdm(range(epochs), desc="epoch"):
    train_df = train_df.sample(frac=1).reset_index(drop=True)
    for batch_index in tqdm(range(int(math.ceil(train_df.shape[0]/batch_size))), desc="batch", leave=False):
        tamanho_batch_atual = min((batch_index+1)*batch_size, train_df.shape[0]) - batch_index*batch_size
        z = tf.random.normal((tamanho_batch_atual,4,4,8))
        with tf.GradientTape(persistent=True) as tape:
            imagens_geradas = gerador(z)
            # print(f"Escala imagens geradas: {tf.reduce_min(imagens_geradas)} -> {tf.reduce_max(imagens_geradas)}")
            imagens_reais = images_from_df(range(batch_index*batch_size, min((batch_index+1)*batch_size, train_df.shape[0])), df=train_df)
            imagens_reais += tf.random.normal(
                imagens_reais.shape,
                stddev=0.05
            )
            imagens_reais = tf.clip_by_value(
                imagens_reais,
                -1.0,
                1.0
            )
            # print(f"Escala imagens reais: {tf.reduce_min(imagens_reais)} -> {tf.reduce_max(imagens_reais)}")
            y_pred_gerado = critico(imagens_geradas)
            y_pred_real = critico(imagens_reais)
            epsilon = tf.random.uniform(
                [tamanho_batch_atual,1,1,1]
            )

            x_hat = (
                epsilon*imagens_reais
                +
                (1-epsilon)*imagens_geradas
            )
            with tf.GradientTape() as gp_tape:
                gp_tape.watch(x_hat)

                pred = critico(x_hat)
            grad = gp_tape.gradient(pred, x_hat)
            norm = tf.sqrt(
                tf.reduce_sum(
                    tf.square(grad),
                    axis=[1,2,3]
                )
            )
            gp = tf.reduce_mean(
                (norm - 1.0) ** 2
            )

            loss_critico = tf.reduce_mean(y_pred_gerado) - tf.reduce_mean(y_pred_real) + lambda_gp*gp
            loss_history.append(loss_critico.numpy())
            loss_gerador = -tf.reduce_mean(y_pred_gerado)
            # gerando mais imagens para treinar o gerador 2 vezes a cada vez que o critico treina para ver se as losses se equilibram
            z = tf.random.normal((tamanho_batch_atual,4,4,8))
            imagens_geradas = gerador(z)
            y_pred_gerado = critico(imagens_geradas)
            loss_gerador = -tf.reduce_mean(y_pred_gerado)

        grads_critico = tape.gradient(loss_critico, critico.trainable_variables)
        optimizer_critico.apply_gradients(
            zip(grads_critico, critico.trainable_variables)
        )
        # if batch_index % 2 == 0:
        grads_gerador = tape.gradient(loss_gerador, gerador.trainable_variables)
        for i, g in enumerate(grads_gerador):
            if g is not None:
                if i in grads_history.keys():
                    grads_history[i].append(tf.reduce_mean(tf.abs(g)).numpy())
                else:
                    grads_history[i] = [tf.reduce_mean(tf.abs(g)).numpy()]
        if batch_index % 67 == 0:
            ckpt.epoch.assign(epoch)
            ckpt.batch.assign(batch_index)

            save_path = manager.save()

            print("Checkpoint salvo em:", save_path)
        optimizer_gerador.apply_gradients(
            zip(grads_gerador, gerador.trainable_variables)
        )

In [ ]:
plt.figure()
plt.suptitle("Gradients")
for indice, valores in grads_history.items():
    plt.plot(valores)
plt.show()
plt.figure()
plt.suptitle("Losses")
plt.plot(loss_history, label='loss')
plt.show()

In [ ]:
z = tf.random.uniform((10,4,4,8))
imagens = gerador(z)
for i in range(10):
    plt.figure()
    plt.imshow(tf.squeeze(imagens[i]), cmap="gray")